In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, cross_val_score 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

import joblib
import dill

In [2]:
class TimeCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, df):
        df = df.copy()

        df.drop_duplicates(inplace=True)

        mask = df["Time"].astype(str).str.contains(", 24:")
        df.loc[mask, "Time"] = (
            df.loc[mask, "Time"].str.replace(", 24:", ", 00:", n=1)
        )

        df["Time"] = pd.to_datetime(df["Time"], format="%d/%m/%Y, %H:%M:%S")
        df = df.sort_values("Time").set_index("Time")

        return df

In [3]:
class KWHProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, gap_threshold=1.0, max_synthetic_points=10):
        self.gap_threshold = gap_threshold
        self.max_synthetic_points = max_synthetic_points

    def fit(self, X, y=None):
        return self

    def split_large_gaps(self, df):
        """Split gaps into evenly distributed synthetic points"""
        rows_to_add = []
        gap_hours = df.index.to_series().diff().dt.total_seconds() / 3600

        for i in range(1, len(df)):
            if gap_hours.iloc[i] > self.gap_threshold:
                prev_time = df.index[i - 1]
                curr_time = df.index[i]
                total_kwh = df.iloc[i]["KWH_diff"]
                
                # Calculate number of splits needed
                gap_duration = (curr_time - prev_time).total_seconds() / 3600
                num_splits = min(
                    int(np.ceil(gap_duration / self.gap_threshold)),
                    self.max_synthetic_points
                )
                
                # Distribute energy evenly across all segments
                kwh_per_segment = total_kwh / num_splits
                
                # Update the current row with its share
                df.at[curr_time, "KWH_diff"] = kwh_per_segment
                
                # Mark current row as synthetic (it now contains interpolated data)
                df.at[curr_time, "is_synthetic"] = 1
                
                # Create synthetic points
                for j in range(1, num_splits):
                    synthetic_time = prev_time + (curr_time - prev_time) * (j / num_splits)
                    
                    new_row = df.loc[curr_time].copy()
                    new_row.name = synthetic_time
                    new_row["KWH_diff"] = kwh_per_segment
                    new_row["is_synthetic"] = 1  # Mark as synthetic
                    
                    # Set electrical measurements to NaN for synthetic points
                    for col in ["AVG_CURRENT", "AVG_V_LL", "AVG_V_LN", "FREQUENCY"]:
                        if col in new_row:
                            new_row[col] = np.nan
                    
                    rows_to_add.append(new_row)

        if rows_to_add:
            df = pd.concat([df, pd.DataFrame(rows_to_add)]).sort_index()

        return df

    def transform(self, df):
        df = df.copy()

        df["KWH_diff"] = df["TOTAL_NET_KWH"].diff()
        df["KWH_diff"] = df["KWH_diff"].where(df["KWH_diff"] >= 0)
        df = df.dropna(subset=["KWH_diff"])

        # Single pass - no while loop needed
        df = self.split_large_gaps(df)

        return df

In [4]:
class HourlyResampler(BaseEstimator, TransformerMixin):
    def __init__(self, fill_method='forward', max_fill_hours=3):
        """
        Parameters:
        -----------
        fill_method : str
            'forward', 'interpolate', or 'none'
        max_fill_hours : int
            Maximum hours to forward-fill missing electrical measurements
        """
        self.fill_method = fill_method
        self.max_fill_hours = max_fill_hours
    
    def fit(self, X, y=None):
        return self

    def transform(self, df):
        hourly = (
            df.resample("1H")
            .agg({
                "KWH_diff": "sum",
                "AVG_CURRENT": "mean",
                "AVG_V_LN": "mean"
            })
        )

        hourly.rename(columns={"KWH_diff": "HOURLY_KWH"}, inplace=True)
        hourly["HOURLY_KWH"].replace(0.0, np.nan, inplace=True)

        # Handle electrical measurements
        electrical_cols = ["AVG_CURRENT", "AVG_V_LL", "AVG_V_LN", "FREQUENCY"]
        
        if self.fill_method == 'forward':
            # Forward fill but only for short gaps
            for col in electrical_cols:
                if col in hourly.columns:
                    hourly[col] = hourly[col].fillna(method='ffill', limit=self.max_fill_hours)
                    # Remaining NaNs get median (for long gaps)
                    hourly[col] = hourly[col].fillna(hourly[col].median())
        
        elif self.fill_method == 'interpolate':
            # Linear interpolation for short gaps
            for col in electrical_cols:
                if col in hourly.columns:
                    # Interpolate up to max_fill_hours
                    hourly[col] = hourly[col].interpolate(
                        method='linear', 
                        limit=self.max_fill_hours,
                        limit_direction='both'
                    )
                    # Fill remaining with median
                    hourly[col] = hourly[col].fillna(hourly[col].median())
        
        else:  # 'none'
            # Don't fill - let downstream handle it
            pass


        return hourly


In [5]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, df):
        df = df.copy()

        df["power_proxy"] = df["AVG_CURRENT"] * df["AVG_V_LN"]

        # Basic time components
        df["hour"] = df.index.hour
        df["weekday"] = df.index.weekday
        df["day_of_month"] = df.index.day
        df["month"] = df.index.month
        df["week_of_year"] = df.index.isocalendar().week

        # Cyclic encoding (prevents discontinuity at boundaries)
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        
        df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
        df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)
        
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

        # ============= LAG FEATURES =============
        lag_hours = [1, 2, 24, 168]
        for lag in lag_hours:
            df[f"kwh_lag_{lag}"] = df["HOURLY_KWH"].shift(lag)

        # ============= ROLLING STATISTICS =============
        # Short-term (helps capture immediate trends)
        df["kwh_roll_3h_mean"] = df["HOURLY_KWH"].rolling(3, min_periods=1).mean()
        
        # Daily patterns (24 hours)
        df["kwh_roll_24h_mean"] = df["HOURLY_KWH"].rolling(24, min_periods=12).mean()
        df["kwh_roll_24h_std"] = df["HOURLY_KWH"].rolling(24, min_periods=12).std()
        df["kwh_roll_24h_min"] = df["HOURLY_KWH"].rolling(24, min_periods=12).min()
        df["kwh_roll_24h_max"] = df["HOURLY_KWH"].rolling(24, min_periods=12).max()
        
        # Weekly patterns (168 hours)
        df["kwh_roll_168h_mean"] = df["HOURLY_KWH"].rolling(168, min_periods=84).mean()
        df["kwh_roll_168h_std"] = df["HOURLY_KWH"].rolling(168, min_periods=84).std()
        
        
        # Ratio to moving average (deviation from normal)
        df["kwh_ratio_to_24h_avg"] = df["HOURLY_KWH"] / (df["kwh_roll_24h_mean"] + 1e-6)
        df["kwh_ratio_to_168h_avg"] = df["HOURLY_KWH"] / (df["kwh_roll_168h_mean"] + 1e-6)

        return df


In [6]:
class TargetCapper(BaseEstimator, TransformerMixin):
    def __init__(self, target_col="HOURLY_KWH"):
        self.target_col = target_col

    def fit(self, X, y=None):
        y_series = X[self.target_col]
        Q1 = y_series.quantile(0.25)
        Q3 = y_series.quantile(0.75)
        IQR = Q3 - Q1
        self.lower_ = Q1 - 1.5 * IQR
        self.upper_ = Q3 + 1.5 * IQR
        return self

    def transform(self, df):
        df = df.copy()
        df[self.target_col + "_capped"] = df[self.target_col].clip(
            lower=self.lower_, upper=self.upper_
        )
        return df

In [7]:
class SmartOutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, target_col="HOURLY_KWH"):
        self.target_col = target_col

    def fit(self, X, y=None):
        # Calculate bounds per hour (energy usage varies by time of day)
        hourly_stats = X.groupby(X.index.hour)[self.target_col].agg(['mean', 'std'])
        self.hourly_mean_ = hourly_stats['mean']
        self.hourly_std_ = hourly_stats['std']
        return self

    def transform(self, df):
        df = df.copy()
        
        # Get expected range for each hour
        hour = df.index.hour
        expected_mean = self.hourly_mean_[hour].values
        expected_std = self.hourly_std_[hour].values
        
        # Flag outliers (3 sigma rule, but per hour)
        lower_bound = expected_mean - 3 * expected_std
        upper_bound = expected_mean + 3 * expected_std
        
        df["is_anomaly"] = (
            (df[self.target_col] < lower_bound) | 
            (df[self.target_col] > upper_bound)
        ).astype(int)
        
        # Optional: Z-score relative to hour-of-day pattern
        df["hourly_zscore"] = (df[self.target_col] - expected_mean) / (expected_std + 1e-6)
        
        return df

In [8]:
class DropNullAndTrimDataTill12(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self


    def transform(self, df):
        df = df.copy()

        df = df.dropna()
        cutoff = pd.Timestamp("2025-12-29 00:00:00")
        df = df.loc[df.index <= cutoff]
        return df
        



In [9]:
preprocessing_pipeline = Pipeline([
    ("time_cleaning", TimeCleaner()),
    ("kwh_processing", KWHProcessor()),
    ("resampling", HourlyResampler()),
    ("feature_engineering", FeatureEngineer()),
    ("target_capper", TargetCapper()),
    ("drop_null_trim_till_12am_29Dec", DropNullAndTrimDataTill12())
])

In [10]:
df = pd.read_excel(r"..\Data\Original_Data\Motot_Type_YWNC-203.xlsx")

final_df = preprocessing_pipeline.fit_transform(df)

C:\Users\devan\AppData\Local\Temp\ipykernel_23164\2912701881.py:19: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df.resample("1H")
C:\Users\devan\AppData\Local\Temp\ipykernel_23164\2912701881.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  hourly["HOURLY_KWH"].replace(0.0, np.nan, inplace=True)
C:\Users\devan\AppData\Local\Temp\ipykernel_23164\2912701881.py:37: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill()

In [11]:
final_df.to_csv("transformed_YWNC2cone.csv", index=True)

In [18]:
df = final_df.copy()
df.index = pd.to_datetime(df.index)

# Weekly rolling mean (7 days * 24 hours)
df["weekly_trend"] = df["HOURLY_KWH"].rolling(window=168, min_periods=1).mean()

fig = go.Figure()

# Scatter points
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["HOURLY_KWH"],
    mode="markers",
    name="HOURLY_KWH",
    marker=dict(size=5),
    hovertemplate="Time: %{x}<br>KWH: %{y}<extra></extra>"
))

# Weekly trend line
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["weekly_trend"],
    mode="lines",
    name="Weekly Trend (168h mean)",
    line=dict(width=3),
    hovertemplate="Time: %{x}<br>Weekly Avg: %{y}<extra></extra>"
))

# ---- Add weekly vertical markers (Mondays) ----
week_starts = df.resample("W-MON").first().index

for w in week_starts:
    fig.add_vline(
        x=w,
        line_width=1,
        line_dash="dot",
        opacity=0.4
    )

fig.update_layout(
    title="HOURLY_KWH vs Time with Weekly Markers",
    xaxis_title="Datetime",
    yaxis_title="HOURLY_KWH",
    template="plotly_white",
    hovermode="x unified",
    height=520
)

fig.show()

In [20]:
final_df.loc["2025-08-27":"2025-09-01"]

,HOURLY_KWH,AVG_CURRENT,AVG_V_LN,power_proxy,hour,weekday,day_of_month,month,week_of_year,hour_sin,...,kwh_roll_3h_mean,kwh_roll_24h_mean,kwh_roll_24h_std,kwh_roll_24h_min,kwh_roll_24h_max,kwh_roll_168h_mean,kwh_roll_168h_std,kwh_ratio_to_24h_avg,kwh_ratio_to_168h_avg,HOURLY_KWH_capped


In [12]:
# rf_best = RandomForestRegressor(
#     n_estimators=200,
#     max_features="sqrt",
#     min_samples_split=5,
#     min_samples_leaf=2,
#     random_state=42,
#     n_jobs=-1
# )

In [13]:
# full_pipeline = Pipeline([
#     ("preprocess", preprocessing_pipeline),
#     ("rf", rf_best)   # Your trained / configured RandomForestRegressor
# ])

In [14]:
# joblib.dump(full_pipeline, "full_pipeline.pkl")

In [15]:
# with open("full_pipeline_dill.pkl", "wb") as f:
#     dill.dump(full_pipeline, f)

In [16]:
# full_pipeline.fit(final_df.drop(columns=["HOURLY_KWH_capped", "HOURLY_KWH"], errors="ignore"), final_df["HOURLY_KWH_capped"])

In [17]:
# df = pd.read_excel(r"Data\Original_Data\Motot_Type_YWNC-203.xlsx")

# final_df = full_pipeline.named_steps["preprocess"].transform(df)